In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

## Random Forest Regression

In [ ]:
df=pd.read_csv(r"C:\Users\goex1\Desktop\DS Projekte f?r GITHUB\Mobile Phone\smartphones_data.csv.csv")

#overview of dataset
df.head(5)


In [ ]:
print("rows and columns in dataset:", df.shape)

## Build Pipeline for categorial and numerical features

In [ ]:
df_metric=df[['RAM', 'storage', 'Num_Rear_Cameras', 'primery_front_camera', 'num_front_camera', 'display_size(inch)','refresh_rate(hz)']]
df_categoric=df[['OS','has_fast_charging','has_fingerprints','has_nfc','has_5g','processor_brand']]

In [ ]:
numerical_pipeline= Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Schritt 1 bei numerischen Variablen
    ('scaler', StandardScaler())                    # Schritt 2 bei numerischen Variablen
])

categorical_pipeline= Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Schritt 1 bei kategorischen Variablen
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # Schritt 2 bei kategorischen Variablen
])

## Join Pipelines with Columntransformer

In [ ]:
preprocessor=ColumnTransformer([('num', numerical_pipeline,df_metric.columns),('cat', categorical_pipeline, df_categoric.columns)])

In [ ]:
Pipeline1=Pipeline([('preprocessor', preprocessor),
                    ('model', RandomForestRegressor())
])

In [ ]:
X = df[df_metric.columns.tolist() + df_categoric.columns.tolist()]
y=df['Price']

In [ ]:
df_metric.corrwith(y).sort_values(ascending=False)

## Train test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
Pipeline1.fit(X_train,y_train)

## Prediction

In [ ]:
y_prediction=Pipeline1.predict(X_test)



## feature importance


In [ ]:

# Feature Importances abrufen
importances = Pipeline1.named_steps['model'].feature_importances_

# Feature-Namen abrufen (nach Preprocessing)
feature_names = Pipeline1.named_steps['preprocessor'].get_feature_names_out()

# DataFrame erstellen
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(5)  # Top 5 Features


## Visual of feature importance

In [ ]:

plt.figure(figsize=(10,6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(5), color='silver')
plt.title("Top 5 Feature Importances for Price")
plt.show()


## Model Performance

In [ ]:

print("R?:", r2_score(y_test, y_prediction))
print ("MAE:", mean_absolute_error(y_test,y_prediction))
print ("MSE:", mean_squared_error(y_test,y_prediction))


## Linear Regression

In [ ]:
Pipeline2=Pipeline([('preprocessor', preprocessor),
                    ('model', LinearRegression())
])

In [ ]:
Pipeline2.fit(X_train,y_train)

In [ ]:
y_prediction_lr=Pipeline2.predict(X_test)

In [ ]:
coefficients = Pipeline2.named_steps['model'].coef_
intercept = Pipeline2.named_steps['model'].intercept_

print(coefficients)
print (intercept)

In [ ]:
print ("MSE=", mean_squared_error(y_test,y_prediction_lr))
print ("MAE=", mean_absolute_error(y_test,y_prediction_lr))
print ("R?=",r2_score(y_test,y_prediction_lr))

## Compare Linear Regression vs Random Forest

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error

# Zwei Subplots nebeneinander
fig, axes = plt.subplots(1, 2, figsize=(16, 7))  # 1 Reihe, 2 Spalten

# --- Lineare Regression ---
axes[0].scatter(y_test, y_prediction_lr, color='blue', alpha=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[0].set_xlabel('Echte Werte')
axes[0].set_ylabel('Vorhersagen')
axes[0].set_title('Linear Regression\nR?: {:.2f}, MAE: {:.0f}'.format(
    r2_score(y_test, y_prediction_lr),
    mean_absolute_error(y_test, y_prediction_lr)
))

# --- Random Forest ---
axes[1].scatter(y_test, y_prediction, color='red', alpha=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
axes[1].set_xlabel('Echte Werte')
axes[1].set_ylabel('Vorhersagen')
axes[1].set_title('Random Forest\nR?: {:.2f}, MAE: {:.0f}'.format(
    r2_score(y_test, y_prediction),
    mean_absolute_error(y_test, y_prediction)
))

plt.tight_layout()
plt.show()


## Random Forest Regression vs. Linear Regression

| Modell             | R?    | MAE   |
| ------------------ | ----- | ----- |
| Lineare Regression | 0.70 | 7178 |
| Random Forest      | 0.80 | 4931 | 


Conlcusion: Random Forest has better performance in this case!